In [1]:
# DEA (CCR input-oriented) using scipy.optimize.linprog
import numpy as np
import pandas as pd

In [3]:


try:
    from scipy.optimize import linprog
except Exception as e:
    raise RuntimeError(f"SciPy not available: {e}")

# Define DMUs (schools) with two inputs and two outputs
schools = ["A", "B", "C"]
X = np.array([  # inputs: [teachers, budget]
    [10, 50],
    [15, 60],
    [20, 90],
], dtype=float)

Y = np.array([  # outputs: [graduates, test score avg]
    [100, 80],
    [120, 70],
    [130, 85],
], dtype=float)

n, m = X.shape[0], X.shape[1]   # DMUs, inputs
s = Y.shape[1]                  # outputs

eff_scores = []
lambdas_all = []
slacks_input = []
slacks_output = []

for j in range(n):
    x0 = X[j]
    y0 = Y[j]
    
    # Decision variables: [theta, lambda_1..lambda_n, s- (m inputs), s+ (s outputs)]
    # We'll formulate standard CCR without explicit slacks in objective: min theta
    # Subject to:
    #   X^T * lambda <= theta * x0        (m constraints)
    #   Y^T * lambda >= y0                (s constraints)
    #   lambda >= 0
    # We'll solve min theta by linear programming by converting to standard form:
    # variables: theta (scalar) + lambda (n)
    
    # Objective: minimize theta
    c = np.zeros(1 + n)
    c[0] = 1.0  # theta coeff
    
    # Inequality constraints A_ub x <= b_ub
    # X^T * lambda - theta * x0 <= 0  ->  (-x0)*theta + (X^T)*lambda <= 0 for each input
    A_ub = np.zeros((m, 1 + n))
    b_ub = np.zeros(m)
    for i in range(m):
        A_ub[i, 0] = -x0[i]
        A_ub[i, 1:] = X[:, i]
    
    # For Y:  -Y^T * lambda <= -y0 (because Y^T lambda >= y0)
    A_ub2 = np.zeros((s, 1 + n))
    b_ub2 = -y0.copy()
    for r in range(s):
        A_ub2[r, 1:] = -Y[:, r]
    A_ub = np.vstack([A_ub, A_ub2])
    b_ub = np.concatenate([b_ub, b_ub2])
    
    bounds = [(0, None)] * (1 + n)  # theta >= 0, lambda >= 0
    
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
    if not res.success:
        raise RuntimeError(f"LP failed for DMU {schools[j]}: {res.message}")
    
    theta = res.x[0]
    lambdas = res.x[1:]
    eff_scores.append(theta)
    lambdas_all.append(lambdas)
    
    # Compute slacks (not optimized here, but we can compute residuals):
    # input slacks s- = theta*x0 - X^T lambda (nonnegative ideally)
    s_minus = theta * x0 - X.T @ lambdas
    # output slacks s+ = Y^T lambda - y0 (nonnegative ideally)
    s_plus = Y.T @ lambdas - y0
    slacks_input.append(s_minus)
    slacks_output.append(s_plus)

eff_df = pd.DataFrame({
    "School": schools,
    "Efficiency (theta)": np.round(eff_scores, 3),
})
# Add lambdas and slacks for transparency
lambda_df = pd.DataFrame(np.round(np.vstack(lambdas_all), 3), columns=[f"λ_{s}" for s in schools])
sminus_df = pd.DataFrame(np.round(np.vstack(slacks_input), 3), columns=[f"s-_{i+1}" for i in range(m)])
splus_df = pd.DataFrame(np.round(np.vstack(slacks_output), 3), columns=[f"s+_{r+1}" for r in range(s)])

result_df = pd.concat([eff_df, lambda_df, sminus_df, splus_df], axis=1)

In [4]:
result_df

,School,Efficiency (theta),λ_A,λ_B,λ_C,s-_1,s-_2,s+_1,s+_2
0,A,1.000,1.0,0.0,0.0,0.000,0.0,0.0,0.0
1,B,1.000,1.2,0.0,0.0,3.000,0.0,0.0,26.0
2,C,0.722,1.3,0.0,0.0,1.444,0.0,0.0,19.0
